# 01 — Данные
> **v2 | Агент удержания полосы + ПИД-регулятор | ветка: `v2-agent-pid`**

> 🕐 Обновлён: 2026-04-09 23:38 МСК

**Время:** ~5 минут  
Загружаем 3 000 сэмплов из Udacity, кэшируем в `/content/data/cache.npy` (локальный SSD).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Явно переходим в папку проекта
REPO_PATH = '/content/drive/MyDrive/diploma'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

# Создаём нужные папки на Drive если их нет
os.makedirs('outputs', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.dataset_v2 import load_and_cache, angle_to_label, THRESHOLD, N_BINS

In [ ]:
# Распаковка датасета в локальный /content/data/ (быстро — SSD, не Drive)
import zipfile

ZIP_PATH   = '/content/drive/MyDrive/diploma/data/dataset.zip'  # читаем с Drive
LOCAL_DATA = '/content/data'                                      # пишем локально
IMG_DIR    = f'{LOCAL_DATA}/IMG'

os.makedirs(LOCAL_DATA, exist_ok=True)

if not os.path.exists(IMG_DIR):
    print('Распаковываю в локальный /content/data/ ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(LOCAL_DATA)
    print('Готово')
else:
    print('Уже распакован')

# Ищем CSV локально
import glob
csvs = glob.glob(f'{LOCAL_DATA}/**/*.csv', recursive=True)
print('Найдены CSV:', csvs)

In [ ]:
CSV_PATH = csvs[0]   # первый найденный CSV
print(f'Используем: {CSV_PATH}')
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)

In [ ]:
# ── Диагностика путей + сброс кэша ──────────────────────────────
import pathlib, os

print("=== Проверка путей к изображениям ===")
_df_check = pd.read_csv(CSV_PATH)
_img_col = _df_check.columns[0]
_images_dir_p = pathlib.Path(CSV_PATH).parent

for _, _row in _df_check.head(3).iterrows():
    _raw_str = str(_row[_img_col]).strip()
    _fname   = _raw_str.replace("\\", "/").split("/")[-1]
    _raw     = pathlib.Path(_raw_str)
    _cands   = [
        _images_dir_p / _raw,
        _images_dir_p / _fname,
        _images_dir_p / "IMG" / _fname,
    ]
    _found = [str(p) for p in _cands if p.exists()]
    print(f"  CSV: {_raw_str!r}  →  fname={_fname!r}")
    print(f"  Найдено: {_found if _found else 'НЕТ'}")
    for _c in _cands:
        print(f"    {'OK' if _c.exists() else 'xx'} {_c}")
    print()

# Удаляем кэш если он пуст или не существует
_cache = pathlib.Path('/content/data/cache.npy')
if _cache.exists():
    import numpy as _np
    _d = _np.load(_cache, allow_pickle=True).item()
    if len(_d.get('images', [])) == 0:
        _cache.unlink()
        print("Кэш был пустым — удалён, пересоздастся.")
    else:
        print(f"Кэш ОК: {len(_d['images'])} изображений.")
else:
    print("Кэш не найден — будет создан.")

In [ ]:
# Исправляем Windows-пути в CSV → относительные пути вида "IMG/filename.jpg"
# Нужно если CSV содержит абсолютные пути вида C:\Users\...\IMG\center_xxx.jpg
import pandas as pd, pathlib

_df  = pd.read_csv(CSV_PATH)
_changed = False

for _col in _df.columns[:3]:   # center, left, right
    if _df[_col].dtype == object:
        _sample = str(_df[_col].iloc[0]).strip()
        if '\\' in _sample or (_sample.startswith('/') and 'IMG' not in _sample.split('/')[-2]):
            _df[_col] = _df[_col].apply(
                lambda x: 'IMG/' + str(x).strip().replace('\\', '/').split('/')[-1]
                if pd.notna(x) else x
            )
            _changed = True

if _changed:
    _fixed = str(pathlib.Path(CSV_PATH).parent) + '/driving_log_fixed.csv'
    _df.to_csv(_fixed, index=False)
    CSV_PATH = _fixed
    print(f"Пути исправлены. Пример: {_df.iloc[0, 0]}")
    print(f"Используем: {CSV_PATH}")
else:
    print("Пути в CSV уже в правильном формате, ничего не меняем.")

In [ ]:
# images_dir = локальная папка где лежит CSV (там же обычно IMG/)
import pathlib
IMAGES_DIR = str(pathlib.Path(CSV_PATH).parent) + '/'

# Загружаем и кэшируем 3k сэмплов (кэш тоже локально)
images, angles = load_and_cache(
    csv_path=CSV_PATH,
    images_dir=IMAGES_DIR,
    cache_path='/content/data/cache.npy'
)
print(f'images: {images.shape}  angles: {angles.shape}')
print(f'Размер кэша в RAM: {images.nbytes / 1e6:.1f} MB')

In [ ]:
# ── Гистограмма углов до/после балансировки ──────────────────────
df_all = pd.read_csv(CSV_PATH)
angle_col = 'steering' if 'steering' in df_all.columns else df_all.columns[3]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_all[angle_col], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(-THRESHOLD, color='red', linestyle='--', label=f'±{THRESHOLD}')
axes[0].axvline( THRESHOLD, color='red', linestyle='--')
axes[0].set_title(f'Полный датасет ({len(df_all):,} сэмплов)')
axes[0].set_xlabel('Угол руля')
axes[0].legend()

axes[1].hist(angles, bins=50, color='darkorange', edgecolor='white')
axes[1].axvline(-THRESHOLD, color='red', linestyle='--', label=f'±{THRESHOLD}')
axes[1].axvline( THRESHOLD, color='red', linestyle='--')
axes[1].set_title(f'После балансировки ({len(angles):,} сэмплов)')
axes[1].set_xlabel('Угол руля')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/histograms.png', dpi=120)
plt.show()

In [ ]:
# ── Примеры изображений с метками ────────────────────────────────
from src.dataset_v2 import angle_to_label, THRESHOLD

labels = np.array([angle_to_label(a) for a in angles])
n_in  = int(labels.sum())
n_out = len(labels) - n_in

if len(labels) == 0:
    print("ОШИБКА: angles пуст! Смотри диагностику выше — изображения не грузятся.")
else:
    print(f'В полосе:   {n_in} ({100*n_in/len(labels):.1f}%)')
    print(f'Вне полосы: {n_out} ({100*n_out/len(labels):.1f}%)')

    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    fig.suptitle('Примеры: зелёный = в полосе, красный = вне полосы')

    for col, (label, color, title) in enumerate([
        (1, 'green', 'В полосе'),
        (0, 'red',   'Вне полосы')
    ]):
        idxs = np.where(labels == label)[0][:6]
        for row, i in enumerate(idxs):
            ax = axes[col][row]
            ax.imshow(images[i, 0], cmap='gray', vmin=-1, vmax=1)
            ax.set_title(f'e={angles[i]:.2f}', fontsize=8, color=color)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig('outputs/sample_images.png', dpi=120)
    plt.show()


In [ ]:
# -- Таблица распределения сэмплов по углам руля -------------------
# Обновлён: 2026-04-28 МСК
import pandas as pd
import numpy as np

df_all = pd.read_csv(CSV_PATH)
angle_col = 'steering' if 'steering' in df_all.columns else df_all.columns[3]

n_bins = 30
bins = np.linspace(-1.0, 1.0, n_bins + 1)
bin_labels = [f'[{bins[i]:.2f}, {bins[i+1]:.2f})' for i in range(n_bins)]

counts_before, _ = np.histogram(df_all[angle_col].values, bins=bins)
counts_after,  _ = np.histogram(angles, bins=bins)

dist_df = pd.DataFrame({
    'Диапазон угла':        bin_labels,
    'До балансировки':      counts_before,
    'После балансировки':   counts_after,
    '% (после)':           (counts_after / max(counts_after.sum(), 1) * 100).round(1),
})
dist_df = dist_df[counts_before > 0].reset_index(drop=True)

print(f'Всего сэмплов до:    {counts_before.sum():,}')
print(f'Всего сэмплов после: {counts_after.sum():,}')
display(dist_df)

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(dist_df) + 1.2)))
ax.axis('off')
tbl = ax.table(
    cellText=dist_df.values,
    colLabels=dist_df.columns,
    cellLoc='center',
    loc='center',
    colColours=['#1F4788'] * 4,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.4)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.get_text().set_color('white')
        cell.get_text().set_fontweight('bold')
    elif col > 0:
        cell.get_text().set_ha('right')
plt.title('Распределение сэмплов по углам руля', pad=10, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/angle_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Сохранено: outputs/angle_distribution.png')